In [0]:
customers_df = spark.table("data_analyst_demo.ecommerce.bronze_customers")

orders_df = spark.table("data_analyst_demo.ecommerce.bronze_orders")

order_items_df = spark.table("data_analyst_demo.ecommerce.bronze_order_items")

product_df = spark.table("data_analyst_demo.ecommerce.bronze_products")

payments_df = spark.table("data_analyst_demo.ecommerce.bronze_order_payments")

reviews_df = spark.table("data_analyst_demo.ecommerce.bronze_order_reviews")

sellers_df = spark.table("data_analyst_demo.ecommerce.bronze_sellers")

In [0]:
%sql
use catalog data_analyst_demo;
use schema ecommerce;

## bronze_customers -----silver_customers

In [0]:
customers_df = spark.table("bronze_customers")

display(customers_df.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:
customers_df.count()

99441

In [0]:
customers_df.count(), customers_df.dropDuplicates().count()

(99441, 99441)

In [0]:
from pyspark.sql.functions import col, sum

customers_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customers_df.columns
]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [0]:
from pyspark.sql.functions import upper

customers_df = customers_df.withColumn(
    "customer_city",
    upper(col("customer_city"))
)

In [0]:
customers_df = (
    customers_df
    .withColumnRenamed("customer_zip_code_prefix","zip_code")
    .withColumnRenamed("customer_city","city")
    .withColumnRenamed("customer_state","state")
)

In [0]:
customers_df.filter(
    col("zip_code").isNull()
).show()

+-----------+------------------+--------+----+-----+
|customer_id|customer_unique_id|zip_code|city|state|
+-----------+------------------+--------+----+-----+
+-----------+------------------+--------+----+-----+



In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")

In [0]:
%sql
SELECT *
FROM silver_customers
LIMIT 10;

customer_id,customer_unique_id,zip_code,city,state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,FRANCA,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,SAO BERNARDO DO CAMPO,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,SAO PAULO,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,MOGI DAS CRUZES,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,CAMPINAS,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,JARAGUA DO SUL,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,SAO PAULO,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,TIMOTEO,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,CURITIBA,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,BELO HORIZONTE,MG


## bronze order ---- silver_order

In [0]:
orders_df = spark.table("bronze_orders")

In [0]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
orders_df = spark.table("bronze_orders")

display(orders_df.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


In [0]:
from pyspark.sql.functions import to_timestamp

orders_df = (
    orders_df
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    )
)

In [0]:
orders_df = orders_df.dropDuplicates()

In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

## bronze_order_item ----- silver_order_items

In [0]:
bronze_order_df = spark.table("bronze_order_items")

display(bronze_order_df.limit(10))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4


In [0]:
order_items_df = spark.table("bronze_order_items")

In [0]:
from pyspark.sql.functions import to_timestamp

order_items_df = order_items_df.withColumn(
    "shipping_limit_date",
    to_timestamp("shipping_limit_date")
)

display(order_items_df.limit(10))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4


In [0]:
order_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_order_items")

## bronze_product ----- silver_product


In [0]:
product_df = spark.table("bronze_products")

display(product_df.limit(10))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12


In [0]:
from pyspark.sql.functions import col, sum

display(
    product_df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in product_df.columns
    ])
)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,610,610,610,610,2,2,2,2


In [0]:
product_df = product_df.dropna()

In [0]:
product_df = product_df.fillna({
    "product_category_name": "Unknown"
})

In [0]:
product_df = product_df.fillna({
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_height_cm": 0,
    "product_width_cm": 0
})

In [0]:
product_df = product_df.dropna(
    subset=[
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght"
    ]
)

In [0]:
product_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in product_df.columns
]).show()

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                    0|                  0|                         0|                 0|               0|                0|                0|               0|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



In [0]:
from pyspark.sql.functions import col, sum

product_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght"
    ]
]).show()

+---------------------+-------------------+--------------------------+
|product_category_name|product_name_lenght|product_description_lenght|
+---------------------+-------------------+--------------------------+
|                    0|                  0|                         0|
+---------------------+-------------------+--------------------------+



In [0]:
product_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_product")

## bronze_payment--- silver_payment

In [0]:
payments_df = spark.table("bronze_payments")

display(payments_df.limit(10))

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [0]:
from pyspark.sql.functions import col, sum

payments_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in payments_df.columns
]).show()

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



In [0]:
payments_df.count()

payments_df.dropDuplicates().count()

103886

In [0]:
payments_df = payments_df.dropDuplicates()

In [0]:
payments_df = payments_df.fillna({
    "payment_type": "Unknown",
    "payment_installments": 0,
    "payment_value": 0.0
})

In [0]:
payments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.silver_payments")

In [0]:
%sql
show tables;

database,tableName,isTemporary
ecommerce,bronze_category_translation,false
ecommerce,bronze_customers,false
ecommerce,bronze_geolocation,false
ecommerce,bronze_order_items,false
ecommerce,bronze_orders,false
ecommerce,bronze_payments,false
ecommerce,bronze_products,false
ecommerce,bronze_reviews,false
ecommerce,bronze_sellers,false
ecommerce,silver_customers,false


## bronze_review----- silver_review

In [0]:
reviews_df_df = spark.table("bronze_reviews")

display(reviews_df_df.limit(10))

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:
from pyspark.sql.functions import to_timestamp

review_df = reviews_df.withColumn(
    "review_creation_date",
    to_timestamp("review_creation_date")
).withColumn(
    "review_answer_timestamp",
    to_timestamp("review_answer_timestamp")
)

In [0]:
review_df = spark.table("bronze_reviews")

In [0]:
display(review_df.limit(10))

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:
from pyspark.sql.functions import col, sum

review_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in review_df.columns
]).show()

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [0]:
from pyspark.sql.functions import to_timestamp

review_df = review_df.withColumn(
    "review_creation_date",
    to_timestamp("review_creation_date")
).withColumn(
    "review_answer_timestamp",
    to_timestamp("review_answer_timestamp")
)

In [0]:
review_df = review_df.fillna({
    "review_comment_title": "No Title",
    "review_comment_message": "No Review"
})

In [0]:
review_df = review_df.dropna(
    subset=[
        "review_id",
        "order_id",
        "review_score"
    ]
)

In [0]:
review_df.filter(col("review_score").isNull()).show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|,2018-02-16 00:00...|                NULL|        NULL|                NULL|                  NULL|                NULL|                   NULL|
|A entrega foi efe...|                NULL|        NULL|                NULL|                  NULL|                NULL|                   NULL|
|O produto já come...|                NULL|        NULL|                NULL|                  NULL|                NULL|                   NULL|
|    Estou satisfeita|                NULL|        NULL|                NULL|                  NULL|                NULL|   

In [0]:
review_df = spark.table("data_analyst_demo.ecommerce.bronze_reviews")

In [0]:
from pyspark.sql.functions import to_timestamp

review_df = review_df.withColumn(
    "review_creation_date",
    to_timestamp("review_creation_date")
).withColumn(
    "review_answer_timestamp",
    to_timestamp("review_answer_timestamp")
)

In [0]:
review_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.silver_reviews")

In [0]:
%sql
show tables;

database,tableName,isTemporary
ecommerce,bronze_category_translation,false
ecommerce,bronze_customers,false
ecommerce,bronze_geolocation,false
ecommerce,bronze_order_items,false
ecommerce,bronze_orders,false
ecommerce,bronze_payments,false
ecommerce,bronze_products,false
ecommerce,bronze_reviews,false
ecommerce,bronze_sellers,false
ecommerce,silver_customers,false


## bronze_seller ----- silver_seller


In [0]:
seller_df = spark.table("bronze_sellers")
display(seller_df.limit(10))




seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


In [0]:
seller_df = spark.table("data_analyst_demo.ecommerce.bronze_sellers")

In [0]:
seller_df.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum

seller_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in seller_df.columns
]).show()

+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



In [0]:
seller_df = seller_df.dropna(
    subset=["seller_id"]
)

In [0]:
seller_df = seller_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import initcap, upper

seller_df = seller_df.withColumn(
    "seller_city",
    initcap(col("seller_city"))
).withColumn(
    "seller_state",
    upper(col("seller_state"))
)

In [0]:
seller_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.silver_sellers")

## bronze geolocation ---- silver geolocation


In [0]:
geolocation_df = spark.table("bronze_geolocation")
display(geolocation_df.limit(10))

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP
1012,-23.547762303364266,-46.63536053788448,são paulo,SP
1047,-23.546273112412678,-46.64122516971552,sao paulo,SP
1013,-23.546923208436723,-46.6342636964915,sao paulo,SP
1029,-23.543769055769133,-46.63427784085132,sao paulo,SP
1011,-23.547639550320632,-46.63603162315495,sao paulo,SP


In [0]:
geolocation_df = spark.table("data_analyst_demo.ecommerce.bronze_geolocation")

In [0]:
from pyspark.sql.functions import col, sum

geolocation_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in geolocation_df.columns
]).show()

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



In [0]:
geolocation_df = geolocation_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import initcap

geolocation_df = geolocation_df.withColumn(
    "geolocation_city",
    initcap(col("geolocation_city"))
)

In [0]:
from pyspark.sql.functions import upper

geolocation_df = geolocation_df.withColumn(
    "geolocation_state",
    upper(col("geolocation_state"))
)

In [0]:
geolocation_df.printSchema()

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

geolocation_df = geolocation_df.filter(
    (col("geolocation_lat").between(-90, 90)) &
    (col("geolocation_lng").between(-180, 180))
)

In [0]:
geolocation_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.silver_geolocation")

## bronze_catagory_translations ----- silver_catagory_translations


In [0]:
catagory_df = spark.table("data_analyst_demo.ecommerce.bronze_category_translation")

In [0]:
catagory_df = spark.table("bronze_category_translation")
display(catagory_df.limit(10))

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


In [0]:
category_df = spark.table("data_analyst_demo.ecommerce.bronze_category_translation")

In [0]:
category_df.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum

category_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in category_df.columns
]).show()

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



In [0]:
category_df = category_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import trim

category_df = category_df.withColumn(
    "product_category_name",
    trim(col("product_category_name"))
).withColumn(
    "product_category_name_english",
    trim(col("product_category_name_english"))
)

In [0]:
from pyspark.sql.functions import regexp_replace

category_df = category_df.withColumn(
    "product_category_name_english",
    regexp_replace(
        col("product_category_name_english"),
        "_",
        " "
    )
)

In [0]:
from pyspark.sql.functions import initcap

category_df = category_df.withColumn(
    "product_category_name_english",
    initcap(
        regexp_replace(col("product_category_name_english"), "_", " ")
    )
)

In [0]:
category_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.silver_category_translation")

In [0]:
%sql
DROP TABLE IF EXISTS data_analyst_demo.ecommerce.silver_product_category_translation;

In [0]:
%sql
show tables;

database,tableName,isTemporary
ecommerce,bronze_category_translation,false
ecommerce,bronze_customers,false
ecommerce,bronze_geolocation,false
ecommerce,bronze_order_items,false
ecommerce,bronze_orders,false
ecommerce,bronze_payments,false
ecommerce,bronze_products,false
ecommerce,bronze_reviews,false
ecommerce,bronze_sellers,false
ecommerce,silver_category_translation,false
